In [11]:
import torch
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from pathlib import Path
from datasets import load_dataset
import json
from PIL import Image
from patch_analysis.utils import *
from patch_analysis.prob_utils import choice_probs_ABC, probs_tensor_to_dicts
from patch_analysis.hook_utils import *
import pandas as pd

In [6]:
IMAGE_DIR = Path("/weka/eickhoff/esx139/flux_inpainting/flux_klein/consistent_set/qwen8/style0/cat_2/polarity0")
IMAGE_VARIANTS = ["original", "female_bg", "male_bg"]

In [7]:
MODEL = "Qwen/Qwen3-VL-8B-Instruct"  # start small; scale later
device = "cuda"

processor = AutoProcessor.from_pretrained(MODEL)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL,
    device_map="auto",
    torch_dtype=torch.bfloat16,
).eval()
IMAGE_TOKEN_ID = model.config.image_token_id
MERGE_SIZE = processor.image_processor.merge_size  #2
PATCH_SIZE = processor.image_processor.patch_size   # 14


Loading checkpoint shards: 100%|██████████| 4/4 [00:01<00:00,  2.41it/s]


In [4]:
ds = load_dataset("ucf-crcv/SB-Bench", split="real")
with open("/weka/eickhoff/esx139/inpainting/experiments/id_to_index.json", "r") as f:
    id_to_index = json.load(f)

In [17]:
sample_id = "03_29_1316_2_03"
frac = 0.33
records = []

def run_experiment(sample_id, frac):    

    for variant in IMAGE_VARIANTS:
        if variant == "original":
            inputs, image_path = get_input(processor, ds, id_to_index, sample_id, IMAGE_DIR, inpaited=False)
        else:
            inputs, image_path = get_input(processor, ds, id_to_index, sample_id, IMAGE_DIR, style=variant, inpaited=True)
        
        image_grid_thw = inputs['image_grid_thw']
        inputs = inputs.to(device)
        original_image = downsample_image(Image.open(image_path).convert("RGB"))
        orig_w, orig_h = original_image.size
        correct_answer = ["A", "B", "C"][ds[id_to_index[sample_id]]["label"]]

        patch_map = build_qwen3vl_patch_to_pixel_map(
            input_ids=inputs["input_ids"][0],
            image_token_id= IMAGE_TOKEN_ID,
            image_grid_thw=image_grid_thw,
            processor=processor,
            original_height=orig_h,
            original_width=orig_w,
        )
        all_image_positions = sorted(patch_map.keys())
        captured, clean_outputs = clean_capture(model, inputs, all_image_positions, generate=False)
        clean_probs = probs_tensor_to_dicts(choice_probs_ABC(clean_outputs, processor, inputs))[0]

        text_inputs = get_textonly_input(processor, ds, id_to_index, sample_id)
        text_only_outputs = model(**text_inputs.to(device))
        text_only_probs = probs_tensor_to_dicts(choice_probs_ABC(text_only_outputs, processor, text_inputs))[0]

        replacements = compute_replacements(captured, strategy="mean")


        t, h_pre, w_pre = image_grid_thw[0].tolist()
        grid_h = h_pre // MERGE_SIZE
        grid_w = w_pre // MERGE_SIZE

        for row, col, tokens in sliding_window_on_grid(
            patch_map, image_grid_thw, frac=frac, merge_size=MERGE_SIZE
        ):
            ablated_outputs = ablated_forward(
                model,
                inputs, 
                selected_positions=tokens,
                replacements=replacements,
                generate=False,
            )

            ablated_probs = probs_tensor_to_dicts(choice_probs_ABC(ablated_outputs, processor, inputs))[0]

            records.append({
                    # identity
                    "sample_id":      sample_id,
                    "variant":        variant,
                    # grid metadata
                    "grid_h":         grid_h,
                    "grid_w":         grid_w,
                    "win_size":       len(set(r for r in range(row, row + round(min(grid_h, grid_w) * frac)))),  
                    "frac":           frac,
                    "stride":         max(1, max(2, round(min(grid_h, grid_w) * frac)) // 2),
                    # window position
                    "win_row":        row,
                    "win_col":        col,
                    # clean baseline
                    "clean_probs":   clean_probs,
                    #text-only baseline
                    "text-only_probs": text_only_probs,
                    # ablated
                    "ablated-probs": ablated_probs,
                    # derived
                    "correct_answer": correct_answer,
                })
            
    return records
            





In [19]:
all_records = []
sample_ids = []

with open('/weka/eickhoff/esx139/consistent_subset_prob_change/changed_ids_by_background_and_group.json', "r") as f:
    obj = json.load(f)
    sample_ids = obj['female_bg_probs']['M']

for sample_id in sample_ids:
    records = run_experiment(sample_id, frac)
    all_records.extend(records)

df = pd.DataFrame(all_records)

In [15]:
df = pd.DataFrame(records)

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5952 entries, 0 to 5951
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   sample_id        5952 non-null   object 
 1   variant          5952 non-null   object 
 2   grid_h           5952 non-null   int64  
 3   grid_w           5952 non-null   int64  
 4   win_size         5952 non-null   int64  
 5   frac             5952 non-null   float64
 6   stride           5952 non-null   int64  
 7   win_row          5952 non-null   int64  
 8   win_col          5952 non-null   int64  
 9   clean_probs      5952 non-null   object 
 10  text-only_probs  5952 non-null   object 
 11  ablated-probs    5952 non-null   object 
 12  correct_answer   5952 non-null   object 
dtypes: float64(1), int64(6), object(6)
memory usage: 604.6+ KB


In [21]:
df.to_parquet("ablation_results.parquet")

In [22]:
df.head()

,sample_id,variant,grid_h,grid_w,win_size,frac,stride,win_row,win_col,clean_probs,text-only_probs,ablated-probs,correct_answer
0,03_29_1316_2_03,original,43,27,9,0.33,4,0,0,"{'A': 0.0002956390380859375, 'B': 3.5285949707...","{'A': 1.0, 'B': 2.384185791015625e-07, 'C': 6....","{'A': 0.000335693359375, 'B': 2.43186950683593...",A
1,03_29_1316_2_03,original,43,27,9,0.33,4,0,4,"{'A': 0.0002956390380859375, 'B': 3.5285949707...","{'A': 1.0, 'B': 2.384185791015625e-07, 'C': 6....","{'A': 0.0002307891845703125, 'B': 3.1232833862...",A
2,03_29_1316_2_03,original,43,27,9,0.33,4,0,8,"{'A': 0.0002956390380859375, 'B': 3.5285949707...","{'A': 1.0, 'B': 2.384185791015625e-07, 'C': 6....","{'A': 0.000431060791015625, 'B': 5.14984130859...",A
3,03_29_1316_2_03,original,43,27,9,0.33,4,0,12,"{'A': 0.0002956390380859375, 'B': 3.5285949707...","{'A': 1.0, 'B': 2.384185791015625e-07, 'C': 6....","{'A': 0.000431060791015625, 'B': 5.81741333007...",A
4,03_29_1316_2_03,original,43,27,9,0.33,4,0,16,"{'A': 0.0002956390380859375, 'B': 3.5285949707...","{'A': 1.0, 'B': 2.384185791015625e-07, 'C': 6....","{'A': 0.00014019012451171875, 'B': 2.431869506...",A


In [27]:
all_records = []
sample_ids = []

sample_ids = [d.name for d in IMAGE_DIR.iterdir() if d.is_dir()]

for sample_id in sample_ids:
    records = run_experiment(sample_id, frac)
    all_records.extend(records)

df = pd.DataFrame(all_records)

In [28]:
df.to_parquet("ablation_results_style0_p0.parquet")